# 07 — MLflow Model Registry: registro e promoção do modelo

**Objetivo.** Registrar o MLP temporal treinado no **Model Registry** do
MLflow e promovê-lo pelo ciclo de vida **staging → production**, cumprindo o
requisito do Tech Challenge: *"Registrar modelo no MLflow Model Registry →
Staging → Production"*.

## Tracking vs. Registry (a distinção que importa)

| | O que guarda | Pergunta que responde |
|---|---|---|
| **Tracking** (runs) | parâmetros, métricas, artefatos de cada execução | "o que foi tentado e com que resultado?" |
| **Model Registry** | modelos **versionados** com ciclo de vida | "qual modelo está em produção agora?" |

Os notebooks 05/06 fazem *tracking* (logam runs). Este notebook fecha a outra
metade: o modelo vira um cidadão de primeira classe, com nome, versão e
aliases de ambiente — e quem consome (API, batch de inferência) passa a pedir
*"o modelo @production"* em vez de apontar para um arquivo solto.

## Nota sobre Staging → Production no MLflow atual

O mecanismo clássico de *stages* foi **descontinuado** no MLflow moderno em
favor de **aliases** — ponteiros nomeados para versões (ex.: `@staging`,
`@production`). Implementamos o fluxo pedido no enunciado com a API atual:
mesma semântica, mecanismo moderno.

> **Padrão do projeto:** este notebook é o protótipo autocontido (como os
> notebooks 01–06). A versão operacional refatorada vive em
> `scripts/register_model.py` — ver seção final.

## 0. Setup

Mesmo padrão dos demais notebooks: paths relativos à raiz do projeto e a
configuração do MLflow lida do `.env` (`load_dotenv`) — o destino pode ser o
servidor local, o docker-compose ou o Cloud Run, sem mudar nada aqui.

In [1]:
import json
import os
from pathlib import Path

import mlflow
import torch
import yaml
from dotenv import load_dotenv
from mlflow import MlflowClient
from torch import nn

PROJECT_ROOT = Path("..").resolve()
load_dotenv(PROJECT_ROOT / ".env")

TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")
EXPERIMENT_NAME = os.getenv(
    "MLFLOW_EXPERIMENT_NAME", "mlp-market-recommender-system-temporal-v1"
)
assert TRACKING_URI, "MLFLOW_TRACKING_URI nao configurado no .env"

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

REGISTERED_MODEL_NAME = "mlp-temporal-recommender"
UNK_INDEX = 0

train_params = yaml.safe_load(
    (PROJECT_ROOT / "params.yaml").read_text(encoding="utf-8")
)["train"]

print(f"tracking: {TRACKING_URI}")
print(f"experimento: {EXPERIMENT_NAME}")
print(f"checkpoint alvo: {train_params['checkpoint_path']}")

tracking: http://localhost:5000
experimento: mlp-market-recommender-system-temporal-v1
checkpoint alvo: models/mlp_temporal_v1/best_model.pt


## 1. Arquitetura do modelo

A mesma classe validada no notebook 06 (embeddings com `padding_idx=0` para o
UNK + camadas densas). Precisamos dela para reconstruir o modelo a partir do
checkpoint — o `load_state_dict(strict=True)` só aceita arquitetura idêntica,
camada a camada.

In [2]:
class MLPRecommender(nn.Module):
    def __init__(
        self,
        embedding_cardinalities,
        embedding_columns,
        embedding_dim,
        numeric_input_dim,
        hidden_dims,
        dropout,
    ):
        super().__init__()
        self.embedding_columns = list(embedding_columns)
        self.embeddings = nn.ModuleList(
            [
                nn.Embedding(
                    num_embeddings=embedding_cardinalities[column],
                    embedding_dim=embedding_dim,
                    padding_idx=UNK_INDEX,
                )
                for column in self.embedding_columns
            ]
        )
        input_dim = len(self.embedding_columns) * embedding_dim + numeric_input_dim
        layers = []
        previous_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend(
                [nn.Linear(previous_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout)]
            )
            previous_dim = hidden_dim
        layers.append(nn.Linear(previous_dim, 1))
        self.mlp = nn.Sequential(*layers)

    def forward(self, categorical_inputs, numeric_inputs):
        embedded_inputs = [
            embedding(categorical_inputs[:, index])
            for index, embedding in enumerate(self.embeddings)
        ]
        features = torch.cat([*embedded_inputs, numeric_inputs], dim=1)
        return self.mlp(features).squeeze(1)

## 2. Carregar o modelo treinado (checkpoint do pipeline DVC)

O checkpoint é o produto do stage `train` (`dvc repro`) e carrega junto a
própria configuração (`experiment_config`, `feature_config`,
cardinalidades) — o modelo se reconstrói a partir do que foi salvo, sem
valores mágicos no notebook.

In [3]:
checkpoint = torch.load(
    PROJECT_ROOT / train_params["checkpoint_path"],
    map_location="cpu",
    weights_only=False,
)

model = MLPRecommender(
    embedding_cardinalities=checkpoint["embedding_cardinalities"],
    embedding_columns=checkpoint["feature_config"]["embedding_columns"],
    embedding_dim=checkpoint["experiment_config"]["embedding_dim"],
    numeric_input_dim=len(checkpoint["feature_config"]["numeric_feature_columns"]),
    hidden_dims=checkpoint["experiment_config"]["hidden_dims"],
    dropout=checkpoint["experiment_config"]["dropout"],
)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.eval()

print(
    f"melhor época: {checkpoint['epoch']} | NDCG@10 val: {checkpoint['best_metric']:.6f}"
)
print(f"parâmetros treináveis: {sum(p.numel() for p in model.parameters()):,}")

melhor época: 2 | NDCG@10 val: 0.499856
parâmetros treináveis: 5,310,657


## 3. Logar a run de registro e registrar a versão

A run de registro documenta *de onde este modelo veio*: hiperparâmetros,
métricas de validação/teste (do `reports/metrics.json` gerado pelo stage
`evaluate`) e o artefato PyTorch completo.

Dois detalhes de implementação:

- **`pip_requirements` explícito** — o MLflow tenta inferir dependências
  chamando o `pip`, que não existe em venvs do uv; declarar é também a
  prática mais reprodutível.
- Registrar cria uma **nova versão** de `mlp-temporal-recommender` — se já
  houver anteriores, a numeração avança (v1 → v2 → ...). É assim que um
  redeploy funciona.

In [4]:
metrics = json.loads(
    (PROJECT_ROOT / "reports" / "metrics.json").read_text(encoding="utf-8")
)

with mlflow.start_run(run_name="register_mlp_temporal"):
    mlflow.log_params(dict(checkpoint["experiment_config"]))
    mlflow.log_metric("best_epoch", checkpoint["epoch"])
    for split in ("validation", "test"):
        for metric_name, value in metrics[split].items():
            mlflow.log_metric(f"{split}_{metric_name}", value)
    mlflow.log_artifact(str(PROJECT_ROOT / "reports" / "metrics.json"), "evaluation")

    torch_version = torch.__version__.split("+")[0]
    model_info = mlflow.pytorch.log_model(
        model, name="model", pip_requirements=[f"torch=={torch_version}"]
    )

registered = mlflow.register_model(model_info.model_uri, REGISTERED_MODEL_NAME)
print(f"modelo: {REGISTERED_MODEL_NAME} | versão registrada: {registered.version}")

2026/07/07 11:10:17 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/07/07 11:10:18 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered model 'mlp-temporal-recommender' already exists. Creating a new version of this model...
2026/07/07 11:10:18 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: mlp-temporal-recommender, version 3


🏃 View run register_mlp_temporal at: http://localhost:5000/#/experiments/1/runs/4ccac9739fd6445199f4b5694de1ff75
🧪 View experiment at: http://localhost:5000/#/experiments/1
modelo: mlp-temporal-recommender | versão registrada: 3


Created version '3' of model 'mlp-temporal-recommender'.


## 4. Promoção: staging → production

A versão recém-registrada
entra em `@staging` (validação/homologação) e, aprovada, recebe `@production`.
A aprovação se apoia nas métricas do stage `evaluate` logadas na run.

In [5]:
client = MlflowClient()

client.set_registered_model_alias(REGISTERED_MODEL_NAME, "staging", registered.version)
print(f"@staging → v{registered.version} (aguardando aprovação...)")

client.set_registered_model_alias(
    REGISTERED_MODEL_NAME, "production", registered.version
)
print(f"@production → v{registered.version} ✔ promovido")

@staging → v3 (aguardando aprovação...)
@production → v3 ✔ promovido


## 5. Verificação independente + consumo por alias

Consultamos o Registry pela API do cliente (a mesma que a UI usa) e fechamos
o ciclo carregando o modelo **pelo alias** — sem versão, sem caminho de
arquivo. É assim que a API de inferência buscará o modelo em produção.

In [6]:
registered_model = client.get_registered_model(REGISTERED_MODEL_NAME)
production_version = client.get_model_version_by_alias(
    REGISTERED_MODEL_NAME, "production"
)
print(
    f"aliases: { ({alias: str(v) for alias, v in registered_model.aliases.items()}) }"
)
print(
    f"@production → versão {production_version.version} | status {production_version.status}"
)

production_model = mlflow.pytorch.load_model(
    f"models:/{REGISTERED_MODEL_NAME}@production"
)
production_model.eval()

smoke_categorical = torch.zeros(3, 4, dtype=torch.long)  # índices UNK (smoke test)
smoke_numeric = torch.zeros(3, 17)
with torch.no_grad():
    scores = production_model(smoke_categorical, smoke_numeric)
print(f"scores do modelo @production: {scores.tolist()}")
print("ciclo completo: pipeline → checkpoint → Registry → consumo por alias ✔")

aliases: {'production': '3', 'staging': '3'}
@production → versão 3 | status READY


scores do modelo @production: [0.002178087830543518, 0.0021781176328659058, 0.002178087830543518]
ciclo completo: pipeline → checkpoint → Registry → consumo por alias ✔


## 6. Conclusão e refatoração

- Requisito **Registry → Staging → Production** implementado e verificado:
  modelo `mlp-temporal-recommender` versionado, aliases `@staging` e
  `@production` aplicados, artefato consumível por alias.
- **Refatoração (padrão do projeto):** este protótipo está consolidado em
  `scripts/register_model.py` — mesma lógica em funções curtas com type
  hints, usando a Factory de `src/models` (mais robusto para consumo externo:
  o artefato guarda a referência da classe em módulo importável, não em
  `__main__` do notebook). Uso: `uv run python -m scripts.register_model`.
- **Portabilidade:** o destino vem do `.env`. Contra o MLflow do
  docker-compose ou do Cloud Run, o procedimento é idêntico — re-executar
  este notebook (ou o script) popula o Registry do ambiente novo.

**Limitação honesta:** o smoke test da seção 5 usa entradas UNK sintéticas —
prova que o artefato carrega e pontua, não a qualidade das predições (essa
prova está no notebook 06 e no stage `evaluate`).